# Test Data, Mocks, and Fixtures
Companion notebook for the **Test Data, Mocks, and Fixtures** section of Chapter 7.

Agent systems are stateful and can behave differently across runs, so isolated, reproducible tests matter even more here. Mock only at the tool boundary -- not the agent's own decision logic -- and keep fixtures small enough that it's obvious what behavior a test is actually checking.

In [1]:
import sys

!cd agent-app && "{sys.executable}" -m pip install --quiet -e ".[test]"

Two small, readable fixtures live in the same `agent-app/tests/conftest.py` shown in notebook 09 -- `order_state` (known starting state) and `fake_order_tool` (a fixed, fake tool response):

```python
@pytest.fixture
def order_state():
    return {
        "user_id": "u-101",
        "intent": "check_order",
        "memory": {},
        "observations": [],
    }


@pytest.fixture
def fake_order_tool():
    return {
        "order_id": "123",
        "status": "shipped",
        "eta": "2026-04-30",
    }
```

The test -- `run_order_agent` is called with the *fixed* tool result, never a live API, so the result is exactly reproducible:

In [2]:
print(open("agent-app/tests/test_orders.py").read())

from support_agent.agents.triage import run_order_agent


def test_order_status_flow(order_state, fake_order_tool):
    result = run_order_agent(
        "Where is order 123?",
        state=order_state,
        order_lookup_result=fake_order_tool,
    )
    assert result["reply"].startswith("Your order")
    assert result["tool_used"] == "order_lookup"



In [3]:
!cd agent-app && "{sys.executable}" -m pytest tests/test_orders.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.2, pluggy-1.6.0 -- C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\output7\agent-app
configfile: pyproject.toml
plugins: asyncio-0.24.0
asyncio: mode=Mode.STRICT, default_loop_scope=None
collecting ... collected 1 item

tests/test_orders.py::test_order_status_flow PASSED                      [100%]

============================== 1 passed in 0.05s ==============================


C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Lib\site-packages\pytest_asyncio\plugin.py:208: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))


Real external systems (network issues, rate limits, changed data, expired credentials) make tests flaky for reasons that have nothing to do with your code. Keep live calls for a separate, smaller integration stage, and let the main suite run on fixtures and mocks like these.

For completeness, the full suite -- smoke tests, order tests, tool tests, and workflow tests -- all pass together:

In [4]:
!cd agent-app && "{sys.executable}" -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.2, pluggy-1.6.0 -- C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\output7\agent-app
configfile: pyproject.toml
plugins: asyncio-0.24.0
asyncio: mode=Mode.STRICT, default_loop_scope=None
collecting ... collected 6 items

tests/test_orders.py::test_order_status_flow PASSED                      [ 16%]
tests/test_smoke.py::test_config_loads PASSED                            [ 33%]
tests/test_smoke.py::test_basic_agent_flow PASSED                        [ 50%]
tests/test_tools.py::test_known_customer_tier PASSED                     [ 66%]
tests/test_tools.py::test_unknown_customer_defaults_to_standard PASSED   [ 83%]
tests/test_workf

C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Lib\site-packages\pytest_asyncio\plugin.py:208: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))
